# SILVER_DOMINIO

Tabelas de domínio padronizadas: descrições em maiúsculas e **sem o código repetido no
início do texto** (ex.: `"27001 1ª Região de Saúde"` → `"1ª REGIÃO DE SAÚDE"`,
`"03-Clínico"` → `"CLÍNICO"`). Há uma linha por (TABELA, CODIGO).

**Origem:** `BRONZE_DOMINIO`.

**Estratégia de carga:** full (`TRUNCATE` + `INSERT /*+ APPEND */`), idempotente.

In [ ]:
import sys
sys.path.append("..")
from dw_utils import recarregar, consultar

In [ ]:
SQL = """
INSERT /*+ APPEND */ INTO SILVER_DOMINIO (TABELA, CODIGO, DESCRICAO)
SELECT TABELA,
       TRIM(CODIGO),
       MAX(UPPER(REGEXP_REPLACE(TRIM(REGEXP_REPLACE(DESCRICAO, '[[:cntrl:]]', ' ')), '^[0-9]+[ -]+', '')))
FROM BRONZE_DOMINIO
WHERE TRIM(CODIGO) IS NOT NULL
GROUP BY TABELA, TRIM(CODIGO)
"""

In [ ]:
recarregar("SILVER_DOMINIO", SQL, origem="BRONZE_DOMINIO")

In [ ]:
consultar("SELECT * FROM SILVER_DOMINIO FETCH FIRST 10 ROWS ONLY")